In [0]:
# Imports
from effodata import ACDS, golden_rules, Joiner, Sifter, Equality, join_on
from kpi_metrics import KPI, AliasMetric, CustomMetric, AliasGroupby, get_metrics, available_metrics, Rollup, Cube
import pyspark.sql.functions as f
from pyspark.sql.types import *
import re
import os
import sys
import time
import upc_input
import datetime as dt
from seg import profile
from pyspark.sql.window import Window
from poirot import SparkManager

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
kpi = KPI(use_sample_mart = False, apply_privacy_filters = True)
acds = ACDS(use_sample_mart = False, apply_privacy_filters = True)

#### No Closed Loop Campaign Summary Tab of KPF Dashboard.
- Contains No Closed Loop Campaigns (Money Services, Kroger Pay, Kroger Wallet, Debit, Credit, SBE)
- Metrics visible are HHs reached, as well as engagement metrics (since we use Dummy UPCs for No Closed Loop Campaigns)
- Should be able to see engagement metrics per channel type in the dashboard by clicking on the "+" button next to the campaign id.

<img src="./No_Closed_Loop_Campaign_Summary_Sample.png" alt="No_Closed_Loop_Campaign_Summary_Sample.png" width="1000"/>

#### No Closed Loop Summary Tab

In [0]:
# Reminder to use a Non-UC enabled cluster when pulling KPM metadata.
media_history_revamped = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')

# Filtering for only No Closed Loop Campaigns (Money Services, Kroger Pay, Kroger Wallet, Debit, Credit, SBE)
media_history_revamped_no_closed_loop = media_history_revamped.withColumn(
    "business_line",
    f.when(
        f.upper(f.col("project_name")).contains("MONEY SERVICES") |
        f.upper(f.col("project_name")).contains(" MS "), "Money Services"
    ).when(
        f.upper(f.col("project_name")).contains(" PAY "), "Kroger Pay"
    ).when(
        f.upper(f.col("project_name")).contains("KROGER WALLET"), "Kroger Wallet"
    ).when(
        f.upper(f.col("project_name")).contains("ACH"), "Debit"
    ).when(
        f.upper(f.col("project_name")).contains("SBE"), "SBE"
    ).when(
        f.upper(f.col("project_name")).contains("CREDIT"), "Credit"
    ).otherwise("")
)

media_history_revamped_no_closed_loop = media_history_revamped_no_closed_loop.filter(
    f.col("business_line").isin(
        "Credit", "Debit", "Kroger Pay", "Kroger Wallet", "Money Services", "SBE"
    )
)

media_history_revamped_no_closed_loop.display()

In [0]:
# Code to generate closed loop campaign tab

# Reminder to use a Non-UC enabled cluster when pulling KPM metadata.

# media metrics
media_metrics = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/media_metrics/campaign/version=v2/source=azure').filter( (f.col('segment') == 'ALL'))
media_metrics = media_metrics.withColumnRenamed("kpm_project_id", "campaign_id")

# Joining on mhtv campaign id, but also on mhtv job id to get only the latest job id
mhtv_mmet = media_history_revamped_no_closed_loop.select(
  "campaign_id", "job_id", "project_name", "manufacturer", "camp_cost", "adjusted_top_performer", "camp_start_date", "camp_end_date", "test_hh_count"
).join(
  media_metrics,
  on=[media_history_revamped_no_closed_loop.campaign_id == media_metrics.campaign_id,
      media_history_revamped_no_closed_loop.job_id == media_metrics.job_id],
  how='inner'
).drop(media_metrics.job_id, media_metrics.campaign_id)

mhtv_mmet.display()

In [0]:
# Filtering for only relevant and recent KPF No Closed Loop Campaigns
kpf_mhtv_mmet = mhtv_mmet.filter(
    (f.col('camp_start_date') >= '2023-01-01') &
    (f.col("manufacturer").isin("Kroger Wallet", "Kroger Personal Finance")) &
    (f.col("modality") == "All Modalities") &
    (f.col("rom") == "Kroger Only")
).dropDuplicates()

# Filtering for only top performer
kpf_mhtv_mmet_top_performer = kpf_mhtv_mmet.filter(f.col("adjusted_top_performer") == "Top-Performer_KRO").dropDuplicates()
kpf_mhtv_mmet_top_performer.display()

In [0]:
# Pivoting Media Metrics Data to get metrics for each campaign type (including NULL for non-applicable metrics)
kpf_mhtv_mmet_top_performer_pivot = kpf_mhtv_mmet_top_performer.filter(
    f.col("metric").isNotNull()
).groupBy(
    "campaign_id", "project_name", "job_id", "camp_cost", "adjusted_top_performer", "metrics_type", "campaign_type", "camp_start_date", "camp_end_date", "test_hh_count"
).pivot("metric").agg(f.first("value"))

kpf_mhtv_mmet_top_performer_pivot.display()

In [0]:
# Filtering Media Metrics for only engagement metrics, which we will display in the No Closed Loop Dashboard 
engagement_metrics = [
    "unique_clicked", "unique_opened", "clickthrough_rate", 
    "open_rate", "total_impressions", "viewability_percent"
]

filtered_kpf_mhtv_mmet_top_performer_pivot = kpf_mhtv_mmet_top_performer_pivot.select(
    "campaign_id", "project_name", "job_id", "camp_cost", "adjusted_top_performer", "metrics_type", "campaign_type", "camp_start_date", "camp_end_date", "test_hh_count", *engagement_metrics
)

display(filtered_kpf_mhtv_mmet_top_performer_pivot)


In [0]:
# Separate XCMs into distinct tactics based on metrics_type column
filtered_kpf_mhtv_mmet_top_performer_pivot = filtered_kpf_mhtv_mmet_top_performer_pivot.withColumn(
    "campaign_type",
    f.when(f.col("campaign_type") == "XCM", 
        f.when(f.col("metrics_type") == "push_metrics", "PUSH")
         .when(f.col("metrics_type") == "email_metrics_EMOD", "EMOD")
         .when(f.col("metrics_type") == "email_metrics_SSE", "SSE")
         .when(f.col("metrics_type") == "offsite_PAND", "PAND")
         .when(f.col("metrics_type") == "offsite_DISPLAY_AD", "DISPLAY_AD")
         .when(f.col("metrics_type") == "offsite_PINT", "PINT")
         .when(f.col("metrics_type") == "offsite_PRV", "PRV")
         .otherwise(f.col("campaign_type")) 
    ).otherwise(f.col("campaign_type"))
).dropDuplicates()

filtered_kpf_mhtv_mmet_top_performer_pivot.display()

In [0]:
bullseye_table = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')
campaign_ids = [row.campaign_id for row in filtered_kpf_mhtv_mmet_top_performer_pivot.select("campaign_id").distinct().collect()]
campaign_info_id_filter = bullseye_table.filter(f.col("kpm_duplicated_id").isin(campaign_ids))

display(campaign_info_id_filter.select("KPM_DUPLICATED_ID", "campaign_type", "NUM_HHS"))

# Use BE table

In [0]:
# User defined melt to unpivot our data (not built into Pyspark)
def melt(df, id_vars, value_vars, var_name = "variable", value_name = "value"):
    n = len(value_vars)
    expr = ", ".join([f"'{c}', {c}" for c in value_vars])
    return df.selectExpr(
        *id_vars,
        f"stack({n}, {expr}) as ({var_name}, {value_name})"
    )

In [0]:
# melt engagment metrics for Jenny to use
campaign_info = ["campaign_id", "project_name", "job_id", "adjusted_top_performer", "campaign_type", "camp_start_date", "camp_end_date"]
# Drop downloads and redemptions, as per discussions w/ Shumaila and Lauren
engagement_metrics = ["test_hh_count", "unique_clicked", "clickthrough_rate", "unique_opened", "open_rate", "total_impressions", "viewability_percent", "camp_cost"]

no_closed_loop_final = melt(filtered_kpf_mhtv_mmet_top_performer_pivot, id_vars=campaign_info,
    value_vars=engagement_metrics, var_name="metric", value_name="value"
).dropDuplicates()

no_closed_loop_final.display()

In [0]:
# Mapping Channel Type Strings exactly in order to join data to mmci table, which contains the correct
# campaign start and end dates per XCM tactic
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

mapping_data = [
    ("PUSH", "Push Notifications"),
    ("DISPLAY_AD", "Display Ad"),
    ("SSE", "Single Subject Email"),
    ("TDC", "Targeted Digital Coupon"),
    ("PAND", "Pandora"),
    ("PRV", "Pre-Roll Video"),
    ("EMOD", "Email Module"),
    ("PINT", "Pinterest")
]
mapping_df = spark.createDataFrame(mapping_data, ["code", "full_name"])

no_closed_loop_mapped = no_closed_loop_final.join(
    f.broadcast(mapping_df),
    no_closed_loop_final.campaign_type == mapping_df.code,
    how="left"
).withColumn("campaign_type", f.coalesce(f.col("full_name"), f.col("campaign_type"))) \
 .drop("code", "full_name")

no_closed_loop = no_closed_loop_mapped.join(
    f.broadcast(mmci.select("KPM_DUPLICATED_ID", "CHANNEL", "CAMP_START_DATE", "CAMP_END_DATE")),
    (no_closed_loop_mapped.campaign_id == mmci.KPM_DUPLICATED_ID) &
    (no_closed_loop_mapped.campaign_type == mmci.CHANNEL),
    how="inner"
).drop(no_closed_loop_mapped.camp_start_date, no_closed_loop_mapped.camp_end_date).select(
  "campaign_id", "project_name", "job_id", "adjusted_top_performer", "campaign_type", "CAMP_START_DATE", "CAMP_END_DATE", "metric", "value")

no_closed_loop = no_closed_loop.filter(f.col('camp_start_date') >= '2023-01-01')
display(no_closed_loop)


# 151900 for example, check metrics. Also if metrics do not exist, make a dash or NA or something. Do not take the average, just keep at a tactic level. Drop Redemptions and Downloads

In [0]:
no_closed_loop.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/no_closed_loop_tab.csv')